# Setup
データ取り込みを行う。 DuckDB 想定。

# Extract

In [ ]:
from pathlib import Path
from llama_index.core import SimpleDirectoryReader
from assistant_agent.loaders import MarkdownReader

# Vault から読み込み
vault_path = Path("../../docs/dataset_website").resolve()
loader = SimpleDirectoryReader(
    input_dir=vault_path,
    recursive=True,
    file_extractor={".md": MarkdownReader()}
)
docs = loader.load_data()

print(f"{len(docs)} 件のノートを読み込みました")

# Load

In [ ]:
import os
import dotenv
from pathlib import Path

dotenv.load_dotenv()
ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

# 保存先ディレクトリを作成
vault_db_path = Path("../../tests/data/vault_db")
vault_db_path.mkdir(exist_ok=True)
vault_db_path = vault_db_path.resolve()

In [ ]:
from pprint import pprint

import sqlalchemy
from sqlalchemy import text
from assistant_agent.entities import VaultUtils
from assistant_agent.entities.duckdb import VaultBase, SampleEntity

path = vault_db_path / "entity.duckdb"
sa_engine = sqlalchemy.create_engine(f"duckdb:///{path}", connect_args={'read_only': False})

# DB へ取り込み
with sa_engine.connect() as sess:
    sess.execute(text("create schema if not exists assets;"))
    sess.commit()

VaultBase.metadata.create_all(sa_engine)
VaultUtils.sync(docs, sa_engine, SampleEntity)
print(f"{len(docs)} 件のノートを読み込みました")

with sa_engine.connect() as sess:
    # sess.execute(text("checkpoint"))
    res = sess.execute(sqlalchemy.select(SampleEntity).limit(10)).all()
    pprint(res)

sa_engine.dispose()

In [ ]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TransformComponent

chunk_size = 800
chunk_overlap = 80
JAPANESE_PARAGRAPH_SEP = "\n\n"

trans: list[TransformComponent] = [
    SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator=JAPANESE_PARAGRAPH_SEP,
    ),
]
pipe = IngestionPipeline(transformations=trans)
res = pipe.run(documents=docs)
print("\n=====================\n".join([item.text for item in res]))  # pyright: ignore[reportAttributeAccessIssue]


In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

from assistant_agent.entities.duckdb import SampleEntity
from assistant_agent.services import VaultSampleRetriever
from assistant_agent.utils.store_context import DuckDBStoreContext

# レトリーバーを作成
store_ctx = DuckDBStoreContext(vault_db_path)
sa_engine = store_ctx.get_engine()
sample_retriever = VaultSampleRetriever(
    "sample_docstore",
    "sample_vectors",
    store_context=store_ctx,
    transformations=[
        SentenceSplitter(
            chunk_size=1024,
            chunk_overlap=200,
            paragraph_separator="\n\n",
        ),
    ],
    embed_model=GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=ENV_GEMINI_API_KEY,
    ),
    embed_dim=3072,
    vault_entity=SampleEntity,
)

In [ ]:
sample_retriever.sync_chunks()

# Retrieval

In [ ]:
query_res = sample_retriever.search_documents("人事・給与", 5)
for item in query_res:
    print(f"{item.score=}")  # pyright: ignore[reportAttributeAccessIssue]
    print(item.text)
    print("==================")

In [ ]:
store_ctx.close()

# ContextRegistry

In [ ]:
from typing import cast
from pathlib import Path

import assistant_agent.services  # noqa: F401  登録発火
from assistant_agent.utils.context import ContextRegistry
from assistant_agent.utils.store_context import DuckDBStoreContext
from assistant_agent.tools.sample import SampleContext

vault_db_path = Path("../../tests/data/vault_db")
vault_db_path.mkdir(exist_ok=True)
vault_db_path = vault_db_path.resolve()

cr_ctx = DuckDBStoreContext(vault_db_path, read_only=True)
ctx = cast(SampleContext, ContextRegistry.build({"sample_retriever": "qwen3emb06b"}, store_ctx=cr_ctx))

In [ ]:
ctx["sample_retriever"].search_documents("ビッグデータ", 5)

In [ ]:
cr_ctx.close()